# headroom-squeez v2: train a small line classifier (free Kaggle GPU)

Trains `jhu-clsp/ettin-encoder-32m` (32M params, ~5x smaller than the current
150M model) as a Squeez *pooled* line classifier on
`KRLabsOrg/tool-output-extraction-swebench`, using Squeez's own training code
at a pinned commit.

**Before running:** Settings → Accelerator → **GPU T4 x2**, and Settings →
**Internet on**. Then *Run All*. Expected time: ~2-3 hours.

At the end, download `/kaggle/working/squeez_pooled_ettin32m.zip` from the
Output tab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -eo pipefail
cd /kaggle/working
git clone -q https://github.com/KRLabsOrg/squeez.git
cd squeez && git checkout -q aa0bd37c4d32ade33f98e4b278801e4f139fbd9e
pip install -q -e . "transformers==5.2.0" "datasets==3.4.1"
python scripts/download_data.py
wc -l data/encoder_*.jsonl

## Train

T4 has no bf16, so fp16. Effective batch 96 (8 x 12), matching Squeez's
ModernBERT recipe; 4096-token context like the reference run.

Runs on a single GPU (see the cell) even when Kaggle assigns two.

In [ ]:
%%bash
set -eo pipefail
# Squeez's compute_loss reads model._line_sep_id, which DataParallel hides;
# Kaggle's "T4 x2" would otherwise wrap the model. One GPU is plenty for 32M.
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/squeez
python -m squeez.encoder.train     --classifier-type pooled     --train-file data/encoder_train.jsonl     --eval-file data/encoder_dev.jsonl     --base-model jhu-clsp/ettin-encoder-32m     --output-dir /kaggle/working/squeez_pooled_ettin32m     --batch-size 8     --gradient-accumulation-steps 12     --max-length 4096     --learning-rate 5e-5     --num-epochs 4     --fp16     --save-steps 200 --eval-steps 200 2>&1 | grep -v -E "it/s]|s/it]" 

## Evaluate on the held-out test split (same 618 examples as our benchmark)

In [ ]:
%%bash
set -eo pipefail
cd /kaggle/working/squeez
python -m squeez.encoder.evaluate     --model-path /kaggle/working/squeez_pooled_ettin32m     --eval-file data/encoder_test.jsonl     --examples-output /kaggle/working/eval_examples_ettin32m.json 2>&1 | tail -40

In [ ]:
%%bash
set -eo pipefail
cd /kaggle/working
rm -rf squeez_pooled_ettin32m/checkpoint-*
zip -qr squeez_pooled_ettin32m.zip squeez_pooled_ettin32m eval_examples_ettin32m.json
ls -lh squeez_pooled_ettin32m.zip